In [1]:
import torch

A1 = torch.zeros(1, 2, 2); A1[0, 0, 0] = 1; A1[0, 1, 1] = 1
A2 = torch.zeros(2, 2, 2); A2[0, 0, 0] = 1; A2[1, 1, 1] = 1
A3 = torch.zeros(2, 2, 1); A3[0, 0, 0] = 1; A3[1, 1, 0] = 1

psi = torch.einsum('asb,btc,cud->stu', A1, A2, A3).reshape(-1) / 2**0.5
print(psi)

tensor([0.7071, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.7071])


In [5]:
psi = torch.zeros(8); psi[0] = psi[7] = 2**-0.5

# Corte 1|23
M1 = psi.reshape(2, 4)
U1, S1, Vh1 = torch.linalg.svd(M1, full_matrices=False)
A1 = U1.reshape(1, 2, -1)            # (left=1, s1, bond1)
rest = torch.diag(S1) @ Vh1          # lo que queda: (bond1, s2·s3)

# Corte 12|3
M2 = rest.reshape(-1, 2)             # (bond1·s2, s3)
U2, S2, Vh2 = torch.linalg.svd(M2, full_matrices=False)
A2 = U2.reshape(A1.shape[2], 2, -1)  # (bond1, s2, bond2)
A3 = (torch.diag(S2) @ Vh2).reshape(-1, 2, 1)  # (bond2, s3, right=1)

print(S1, S2)
print([tuple(a.shape) for a in (A1, A2, A3)])
print(torch.einsum('asb,btc,cud->stu', A1, A2, A3).reshape(-1))

tensor([0.7071, 0.7071]) tensor([0.7071, 0.7071])
[(1, 2, 2), (2, 2, 2), (2, 2, 1)]
tensor([0.7071, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.7071])


In [6]:
for b in range(2):
    for c in range(2):
        term = torch.einsum('s,t,u->stu', A1[0, :, b], A2[b, :, c], A3[c, :, 0]).reshape(-1)
        print(b, c, term)
    

0 0 tensor([0.7071, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
0 1 tensor([0., -0., 0., -0., 0., -0., 0., -0.])
1 0 tensor([0., 0., 0., 0., 0., 0., 0., 0.])
1 1 tensor([0.0000, -0.0000, -0.0000, 0.0000, 0.0000, -0.0000, -0.0000, 0.7071])


In [7]:
X = torch.tensor([[0., 1.], [1., 0.]])
W = X.T.reshape(1, 2, 1, 2)   # bond 1: W[0, in, 0, out] = X[out, in]

phi = torch.einsum('asb,btc,cud,xsyp,ytzq,zuwr->pqr',
                   A1, A2, A3, W, W, W).reshape(-1) / 2**0.5
print(phi)

tensor([0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000])
